# Q&A Loop and an Agent Library


In [2]:
# In this part, we'll create a Q&A loop that uses an agent library to answer questions.

# We'll use the same setup as the previous section, but we'll use an agent library to answer questions.

# Let's start by importing the necessary libraries:

import json
import openai
import gitsource
import minsearch

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

from openai import OpenAI

openai_client = OpenAI()

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [4]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [5]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [6]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [7]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [7]:
question = "How do I create a dahsbord in Evidently?"

In [8]:
user_input = input("tell me a story about anvil and agents")

In [9]:
# Thats the setup from before now complete. 

# Let's create a function that will perform the Q&A loop:

tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions},
]

iteration_number = 1

# Q&A loop
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    message_history.append({"role": "user", "content": user_prompt})

    # tool-call loop
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )

        print(f'iteraration number {iteration_number}...') 
        message_history.extend(response.output)

        has_function_calls = False

        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True

            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)

        iteration_number = iteration_number + 1
        print()

        if not has_function_calls:
            break

# The Q&A loop: - Asks for user input with input('You:') - Breaks if user types "stop" - Appends the user message to history - Runs the tool call loop (inner loop) - Continues until user says "stop"

# The tool call loop (nested inside): - Makes requests to the LLM - Executes any tool calls - Continues until no more tool calls are needed - Returns to the Q&A loop for the next question

# Interactive agents use this two-loop structure.

iteraration number 1...
ASSISTANT: It seems that you didn't include a question. Please provide your query or topic of interest, and I'll help you find the relevant documentation information.

iteraration number 2...
ASSISTANT: I can't provide a story about a unicorn as it falls outside the scope of the documentation knowledge base. However, if you have questions related to the documentation or functionalities, feel free to ask!

iteraration number 3...
ASSISTANT: To address your question about creating a dashboard, I will start by searching for general information about dashboard creation in the documentation. Understanding the steps and components involved is crucial for providing a thorough answer. Let's proceed with the first search. 

### Search Query: "create dashboard"
executing search({"query":"create dashboard"})...

iteraration number 4...
ASSISTANT: From the first search, I found detailed information on how to create a dashboard and manage its components, including adding tab

## Refactor into the Agent Class

Our current code works, but it has some problems: - Everything is in the global scope (tools, functions, variables) - Hard to reuse with different configurations - Difficult to test in isolation - Can't have multiple agents with different behaviors

By creating an Agent class, we can: - Encapsulate all agent-related state and behavior - Create multiple agent instances with different configurations - Reuse the same agent logic across different projects - Test agents in isolation - Share and maintain our agent code more easily



In [ ]:
# Creating the Agent Class

class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)

            has_function_calls = False

            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True

                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)

            iteration_number = iteration_number + 1
            print()

            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": instructions},
        ]

        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break

            message_history = self.loop(user_prompt, message_history)


The Agent class has: 
- __init__: Stores LLM client, model, instructions, and tools 
- make_call: Handles tool execution 
- loop: The tool call loop (returns message history) 
- qna: The Q&A loop that calls loop for each user message

In [11]:
# Now we can create an agent:

agent = Agent(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=tools
)


In [12]:
# Let's test the loop method with a single question:

messages = agent.loop('evidently adshboards')


iteraration number 0...
ASSISTANT: I will start by searching for "dashboards" to find relevant documentation related to various types of dashboards. Understanding the capabilities and features associated with dashboards can provide insight into how they can be effectively utilized.

Let me perform that search.
executing search({"query":"dashboards"})...

iteraration number 1...
ASSISTANT: The search for "dashboards" provided some useful documentation about adding panels and managing dashboards. It details functionalities such as creating tabs, using both a user interface and Python API for adding panels, and provides information about different panel types that can be included in a dashboard. 

Now, I want to dive deeper into specific functionalities by searching for "adding dashboard panels" to understand how to effectively create and customize panels within dashboards. This will help clarify how to utilize the insights from the initial search to construct effective dashboards.

Let's

In [13]:
agent.qna()


iteraration number 0...
ASSISTANT: Hello! How can I assist you today? If you have a question about a specific topic or need information, feel free to ask!

iteraration number 0...
executing search({"query":"agents and dashboards"})...

iteraration number 1...
ASSISTANT: In my first search, I looked up "agents and dashboards" to find specific documentation related to those topics. However, the search results returned metrics related to ranking and recommendation systems rather than information specifically about agents and dashboards.

Next, I want to analyze the search results to determine if they contain any context that could be related to agents (in terms of system functionalities) and clarify if there are specific dashboards mentioned or if I need to perform additional searches for more direct information. 

Let's perform two more searches: one for "agents" and another for "dashboards" to gather more focused information on each topic.
executing search({"query":"agents"})...
executi

# Summary

We just implemented a mini agent framework with: 
- A tool call loop (inner loop for agent reasoning) 
- A Q&A loop (outer loop for user interaction) 
- An Agent class that encapsulates everything

This is the same pattern used by production agent frameworks. 

Understanding how it works from scratch really helps you use these libraries effectively and customize them when needed.

